# Livrable 1 — Modélisation : tournées de livraison (CVRP)

*Groupe N°5 :*
1. Jean Dona
2. Ann-Dany
3. Protais
4. Axel
5. Yannick

*Tuteur :* **Ahmed NAIT CHABANE**

## Plan
1. Contexte et problème
2. Formalisation des données
3. Preuve de complexité
4. Modèle PLNE (CVRP) et extension CVRPTW
5. Vérification de cohérence du modèle
6. Conclusion et suite
7. Références

## 1. Contexte et problème
L'ADEME encourage les solutions de mobilité plus économiques et moins polluantes. La logistique de transport (courrier, livraison, ordures…) est un enjeu majeur : mieux organiser les tournées réduit les kilomètres parcourus, donc les coûts et les émissions.

**Problème étudié (version de base) :** un dépôt, un ensemble de clients à livrer, une flotte de $k$ camions de **même capacité**. Il faut construire des tournées, chacune partant du dépôt et y revenant, de sorte que tous les clients soient servis exactement une fois et que la **durée totale** soit minimale.

Ce problème est connu dans la littérature sous le nom de **CVRP** (*Capacitated Vehicle Routing Problem*), introduit par Dantzig et Ramser (1959).

**Périmètre retenu :** CVRP (niveau B). Les fenêtres de temps (CVRPTW, niveau A) seront traitées comme une **extension** du modèle.

## 2. Formalisation des données

### 2.1 Le réseau
On modélise le réseau par un **graphe orienté complet** $G = (V, A)$ :

- $V = \{0, 1, \dots, n\}$ : ensemble des sommets
  - $0$ : le **dépôt**
  - $C = \{1, \dots, n\}$ : les **clients** ($n$ clients)
- $A = \{(i, j) \mid i, j \in V,\ i \neq j\}$ : les arcs (trajet direct de $i$ vers $j$)

Chaque arc $(i,j) \in A$ est muni d'une **durée de trajet** $t_{ij} \ge 0$ (matrice $T = (t_{ij})$). Dans un réseau routier réel, $t_{ij}$ est la durée du plus court chemin entre $i$ et $j$ : on se ramène ainsi à un graphe complet.

### 2.2 Les demandes et la flotte
- $q_i \in \mathbb{N}$ : **demande** du client $i \in C$ ; on pose $q_0 = 0$
- $K = \{1, \dots, k\}$ : ensemble des **camions** (identiques)
- $Q \in \mathbb{N}$ : **capacité** de chaque camion

### 2.3 Hypothèses
| Hyp. | Énoncé | Justification |
|---|---|---|
| H1 | $1 \le q_i \le Q$ pour tout $i \in C$ | sinon le client n'est servi par aucun camion 
| H2 | Chaque client est servi par **un seul** camion, en **une seule** visite | pas de livraison fractionnée |
| H3 | Chaque tournée commence et finit au dépôt | énoncé du problème |
| H4 | $t_{ij} = t_{ji}$ (cas symétrique) et inégalité triangulaire | vrai pour les instances euclidiennes de VRPLIB |
| H5 | Temps de trajet **constants** (pas de trafic variable) ; temps de service nul | version de base ; le trafic par tranche horaire serait une extension $t_{ij}(\tau)$ |
| H6 | Au plus $k$ tournées (un camion peut rester au dépôt) | flotte homogène |

**Condition nécessaire de faisabilité :** $\displaystyle \sum_{i \in C} q_i \le k\,Q$. Elle n'est pas suffisante : répartir les demandes dans les camions ressemble au problème de *bin packing*, déjà NP-difficile.

### 2.4 Solution et objectif
Une **tournée** est une suite $r = (0, i_1, \dots, i_m, 0)$ avec $i_l \in C$ distincts.

Une **solution** est un ensemble de tournées $S = \{r_1, \dots, r_p\}$, $p \le k$, tel que :

1. les tournées **partitionnent** $C$ (chaque client apparaît dans exactement une tournée) ;
2. chaque tournée respecte la capacité : $\displaystyle \sum_{l=1}^{m} q_{i_l} \le Q$.

**Durée d'une tournée :** $\displaystyle d(r) = t_{0,i_1} + \sum_{l=1}^{m-1} t_{i_l,i_{l+1}} + t_{i_m,0}$

**Objectif :** $\displaystyle \min_{S}\ \sum_{r \in S} d(r)$

### 2.5 Taille de l'instance
Une instance est entièrement décrite par $(n, k, Q, (q_i)_{i \in C}, T)$. Elle contient $O(n^2)$ données (la matrice $T$), ce qui pèse pour des milliers de clients : à garder en tête pour l'implémentation.

## 3. Complexité théorique

### 3.1 Version décision
La complexité se définit sur des problèmes de décision. On associe au CVRP :

> **CVRP-D** — *Entrée :* une instance $(n, k, Q, q, T)$ et un seuil $B$. *Question :* existe-t-il une solution faisable (§2.4) de coût total $\le B$ ?

Si on savait résoudre le problème d'optimisation en temps polynomial, on résoudrait CVRP-D (on compare le coût optimal à $B$). Donc **optimisation au moins aussi difficile que décision**.

### 3.2 CVRP-D $\in$ NP
Le certificat est une solution $S$ (liste de tournées), de taille $O(n)$. On vérifie en temps $O(n)$ :
1. au plus $k$ tournées, chacune commençant et finissant en $0$ ;
2. chaque client apparaît exactement une fois ;
3. la charge de chaque tournée est $\le Q$ ;
4. le coût total est $\le B$.

### 3.3 CVRP-D est NP-difficile : réduction depuis le TSP
On part du **TSP-D** (voyageur de commerce, version décision), NP-complet (Karp, 1972 ; Garey & Johnson, 1979) :

> *Entrée :* $m$ villes, distances $d_{uv}$, seuil $B$. *Question :* existe-t-il un cycle hamiltonien de longueur $\le B$ ?

**Réduction $f$ : TSP-D $\le_p$ CVRP-D.** À une instance de TSP-D on associe :

| Élément CVRP | Valeur |
|---|---|
| dépôt $0$ | une ville quelconque, par exemple la ville 1 |
| clients $C$ | les $m-1$ autres villes ($n = m-1$) |
| durées $t_{ij}$ | $d_{ij}$ |
| demandes $q_i$ | $1$ pour tout client |
| capacité $Q$ | $m - 1$ |
| flotte $k$ | $1$ |
| seuil | $B$ (inchangé) |

**Construction polynomiale** : on recopie la matrice des distances, soit $O(m^2)$.

**Correction.**
- ($\Rightarrow$) Un cycle hamiltonien de longueur $\le B$, lu à partir de la ville 1, est une tournée $(0, i_1, \dots, i_{m-1}, 0)$ qui visite chaque client une fois. Sa charge vaut $m-1 \le Q$ : faisable, de coût $\le B$.
- ($\Leftarrow$) Une solution faisable avec $k=1$ est **une seule** tournée. Comme tous les clients doivent être servis (partition), elle les visite tous exactement une fois : c'est un cycle hamiltonien, de longueur $\le B$.

Donc l'instance TSP-D est positive $\iff$ l'instance CVRP-D est positive. $\square$

**Conclusion :**
- CVRP-D est **NP-complet** (dans NP et NP-difficile) ;
- le CVRP (optimisation) est **NP-difficile** ;
- le TSP est le **cas particulier** $k=1$, capacité non contraignante : le CVRP est une généralisation stricte. Cette preuve est aussi celle de Lenstra & Rinnooy Kan (1981).

### 3.4 Seconde source de difficulté : l'affectation aux camions
Même sans routage, répartir les clients dans les camions est difficile. Si $t_{ij} = 0$ pour tout $(i,j)$ et $B = 0$, CVRP-D demande de **ranger** les demandes $q_i$ dans $k$ camions de capacité $Q$ : c'est le problème de **bin packing** (NP-complet). Le CVRP combine donc TSP (ordre des visites) et bin packing (affectation aux camions).

### 3.5 Conséquences pratiques
- L'espace de recherche croît au moins comme $(n-1)!/2$ pour un seul camion : 20 clients $\approx 10^{17}$ tournées.
- À moins que $\mathrm{P} = \mathrm{NP}$, aucun algorithme exact polynomial n'existe. Les méthodes exactes (PLNE, branch-and-cut) restent limitées à quelques centaines de clients.
- Pour des **milliers de clients**, on se tourne vers les **heuristiques et métaheuristiques**, avec un compromis qualité / temps. Elles seront étudiées dans les boucles suivantes.

## 4. Modèle PLNE

### 4.1 Données
$V = \{0\} \cup C$, $C = \{1,\dots,n\}$, $t_{ij}$, $q_i$, $Q$, $k$ (§2).

### 4.2 Variables de décision
$$x_{ij} = \begin{cases} 1 & \text{si un camion va directement du sommet } i \text{ au sommet } j \\ 0 & \text{sinon} \end{cases} \qquad \forall (i,j) \in A$$

$$u_i \in \mathbb{R}_+ : \text{charge cumulée du camion juste après la livraison du client } i, \qquad \forall i \in C$$

### 4.3 Fonction objectif
$$\min \; \sum_{(i,j) \in A} t_{ij}\, x_{ij} \tag{0}$$

### 4.4 Contraintes
$$\sum_{j \in V,\, j \ne i} x_{ij} = 1 \qquad \forall i \in C \tag{1}$$
$$\sum_{i \in V,\, i \ne j} x_{ij} = 1 \qquad \forall j \in C \tag{2}$$
$$\sum_{j \in C} x_{0j} \le k \tag{3}$$
$$u_i - u_j + Q\, x_{ij} \le Q - q_j \qquad \forall i, j \in C,\ i \ne j \tag{4}$$
$$q_i \le u_i \le Q \qquad \forall i \in C \tag{5}$$
$$x_{ij} \in \{0,1\} \qquad \forall (i,j) \in A \tag{6}$$

### 4.5 Rôle de chaque contrainte
| Contrainte | Signification |
|---|---|
| (1) | chaque client a **exactement un successeur** (un camion en repart) |
| (2) | chaque client a **exactement un prédécesseur** (un camion y arrive) |
| (3) | au plus $k$ camions quittent le dépôt |
| (4) | **capacité + élimination des sous-tours** (voir ci-dessous) |
| (5) | la charge cumulée est au moins la demande du client et au plus $Q$ |
| (6) | variables binaires |

**Pourquoi (4) fonctionne.**
- Si $x_{ij} = 1$ : (4) devient $u_j \ge u_i + q_j$. La charge cumulée augmente de $q_j$ d'un client au suivant ; avec (5), elle ne dépasse jamais $Q$ : **capacité respectée**.
- Si $x_{ij} = 0$ : (4) devient $u_i - u_j \le Q - q_j$, toujours vrai car $u_i \le Q$ et $u_j \ge q_j$ : **contrainte inactive**.
- Un **sous-tour** (cycle $i_1 \to \dots \to i_p \to i_1$ sans le dépôt) donnerait, en sommant $u_j \ge u_i + q_j$ le long du cycle, $0 \ge \sum q_{i_l} > 0$ puisque $q_i \ge 1$ : **impossible**. Tout cycle passe donc par le dépôt.

**Remarque.** Le nombre de départs du dépôt égale le nombre de retours : en sommant (1) et (2) sur $C$, on obtient $\sum_{j\in C} x_{0j} = \sum_{i \in C} x_{i0}$, ce qui est redondant et n'a pas besoin d'être imposé.

### 4.6 Taille du modèle
$n(n+1)$ variables binaires, $n$ variables continues, environ $n^2$ contraintes. Pour $n = 1000$, cela représente environ $10^6$ variables : compact, mais difficile à résoudre exactement.

### 4.7 Extension CVRPTW
Chaque client $i$ a une fenêtre $[a_i, b_i]$ et un temps de service $\sigma_i$ ($\sigma_0 = 0$ ; fenêtre du dépôt $[a_0, b_0]$). On ajoute la variable
$$s_i \in \mathbb{R}_+ : \text{instant de début de service chez } i, \qquad \forall i \in V$$
et les contraintes
$$s_i + \sigma_i + t_{ij} - s_j \le M_{ij}\,(1 - x_{ij}) \qquad \forall (i,j) \in A \tag{7}$$
$$a_i \le s_i \le b_i \qquad \forall i \in V \tag{8}$$
avec $M_{ij} = \max(0,\ b_i + \sigma_i + t_{ij} - a_j)$ (grand-M le plus petit valide).

- (8) **interdit** de livrer hors de la fenêtre ;
- (7) impose $s_j \ge s_i + \sigma_i + t_{ij}$ quand $i \to j$ : on peut **arriver plus tôt et attendre** l'ouverture, puisque $s_j$ peut être strictement supérieur à $s_i + \sigma_i + t_{ij}$ ;
- les variables $s$ éliminent aussi les sous-tours dès que $\sigma_i + t_{ij} > 0$ ; la contrainte (4) reste utile pour la capacité.


## 5. Vérification de cohérence du modèle
On ne **résout pas** le PLNE ici. On vérifie seulement qu'une solution (§2.4) se traduit en $(x, u)$ qui satisfait les contraintes (1) à (6), et qu'un sous-tour ou un dépassement de capacité est bien rejeté.

In [ ]:
def solution_vers_plne(solution, q, n):
    # Traduit des tournées en variables du PLNE : matrice x et charges cumulées u
    x = np.zeros((n + 1, n + 1), dtype=int)
    u = np.zeros(n + 1)
    for r in solution:
        charge = 0
        for a, b in zip(r[:-1], r[1:]):
            x[a, b] = 1
            if b != 0:
                charge += q[b]
                u[b] = charge          # charge cumulée après livraison de b
    return x, u

def contraintes_plne(x, u, T, q, Q, k):
    # Teste (1)-(6). Renvoie un dict {contrainte: bool} et la valeur de l'objectif (0)
    n = len(q) - 1
    C = range(1, n + 1)
    c1 = all(x[i, :].sum() - x[i, i] == 1 for i in C)                  # (1) un successeur
    c2 = all(x[:, j].sum() - x[j, j] == 1 for j in C)                  # (2) un prédécesseur
    c3 = bool(x[0, 1:].sum() <= k)                                           # (3) au plus k camions
    c4 = all(u[i] - u[j] + Q * x[i, j] <= Q - q[j] for i in C for j in C if i != j)   # (4)
    c5 = all(q[i] <= u[i] <= Q for i in C)                             # (5) bornes de charge
    return {"(1)": c1, "(2)": c2, "(3)": c3, "(4)": c4, "(5)": c5}, float((T * x).sum())

# Solution faisable du §2.6 : toutes les contraintes doivent être satisfaites
x, u = solution_vers_plne([[0, 1, 4, 2, 0], [0, 3, 5, 0]], q, n)
print(contraintes_plne(x, u, T, q, Q, k))

In [ ]:
# Cas 1 : un sous-tour (2 -> 3 -> 2) isolé du dépôt. On construit x à la main ;
# aucun vecteur u ne peut satisfaire (4) : on le montre pour u = charges "naturelles".
x_bad = np.zeros((n + 1, n + 1), dtype=int)
x_bad[0, 1] = x_bad[1, 4] = x_bad[4, 5] = x_bad[5, 0] = 1     # tournée 0-1-4-5-0
x_bad[2, 3] = x_bad[3, 2] = 1                                   # sous-tour {2,3}
u_bad = np.array([0, q[1], q[2], q[2] + q[3], q[1] + q[4], q[1] + q[4] + q[5]], dtype=float)
print("sous-tour    :", contraintes_plne(x_bad, u_bad, T, q, Q, k)[0])

# Cas 2 : tournée de charge 4+2+3+5 = 14 > Q = 12 (u_i > Q viole (5))
x_cap, u_cap = solution_vers_plne([[0, 1, 4, 2, 3, 0], [0, 5, 0]], q, n)
print("surcharge    :", contraintes_plne(x_cap, u_cap, T, q, Q, k)[0])

## 6. Conclusion et suite

**Ce que ce livrable établit :**
1. le périmètre : **CVRP** (flotte homogène, capacité $Q$), avec le CVRPTW comme extension modélisée (§4.8) ;
2. une formalisation rigoureuse des données, des solutions et de l'objectif ;
3. la **NP-complétude** de CVRP-D par réduction depuis le TSP : pas d'algorithme exact polynomial connu ;
4. un **PLNE compact** (contraintes (0) à (6)), prêt à être implémenté ;
5. la justification d'une approche **heuristique / métaheuristique** pour les grandes instances.

**Points à valider avec le tuteur :**
- la formulation compacte retenue (§4.1) ;
- l'objectif du CVRPTW : trajet seul ou durée totale (§4.8) ;
- le périmètre : CVRP uniquement, ou CVRPTW dès le livrable final.

**Pour le livrable final :** compléter le PLNE (renforcement, borne inférieure, usage éventuel du solveur exact comme référence sur petites instances), puis choisir et formaliser la métaheuristique.

## 7. Références
- Dantzig, G. B., Ramser, J. H. (1959). The truck dispatching problem. *Management Science*, 6(1), 80–91.
- Karp, R. M. (1972). Reducibility among combinatorial problems. In *Complexity of Computer Computations*, Plenum Press, 85–103.
- Garey, M. R., Johnson, D. S. (1979). *Computers and Intractability: A Guide to the Theory of NP-Completeness*. W. H. Freeman.
- Lenstra, J. K., Rinnooy Kan, A. H. G. (1981). Complexity of vehicle routing and scheduling problems. *Networks*, 11(2), 221–227.
- Kulkarni, R. V., Bhave, P. R. (1985). Integer programming formulations of vehicle routing problems. *European Journal of Operational Research*, 20(1), 58–67.
- Miller, C. E., Tucker, A. W., Zemlin, R. A. (1960). Integer programming formulations and traveling salesman problems. *Journal of the ACM*, 7(4), 326–329.
- Desrochers, M., Laporte, G. (1991). Improvements and extensions to the Miller–Tucker–Zemlin subtour elimination constraints. *Operations Research Letters*, 10(1), 27–36.
- Solomon, M. M. (1987). Algorithms for the vehicle routing and scheduling problems with time window constraints. *Operations Research*, 35(2), 254–265.
- Toth, P., Vigo, D. (2014). *Vehicle Routing: Problems, Methods, and Applications* (2nd ed.). SIAM.
- Uchoa, E. et al. (2017). New benchmark instances for the capacitated vehicle routing problem. *European Journal of Operational Research*, 257(3), 845–858.